# 📘 Домашнє завдання 25. Введення в Agentic AI

Виконав: **Bohdan Pinchuk**

Link: https://github.com/BogdanPinchuk/DataScience-PBY_HW25


## Мета

Створити агента на базі LangGraph, який відповідає на запитання про історію машинного навчання, використовуючи RAG (Retrieval-Augmented Generation).

### Джерело знань

Стаття:

https://arxiv.org/pdf/2408.01747.pdf

**Classical Machine Learning: Seventy Years of Algorithmic Learning Evolution**

---

## Що повинен робити агент?

Користувач ставить запитання:

```text
What are the main milestones in machine learning?
```

або

```text
Which algorithms were influential in the 1990s?
```

Агент:

1. Отримує питання.
2. Шукає релевантні фрагменти статті.
3. Передає знайдений контекст до LLM.
4. Генерує відповідь на основі контексту.

---

## Архітектура

```text
START
  ↓
Retrieve Documents
  ↓
Generate Answer
  ↓
END
```

---

## Використати

### LLM

```python
Qwen/Qwen2.5-1.5B-Instruct
```

### Embeddings

```python
sentence-transformers/all-MiniLM-L6-v2
```

---

## Встановлення

```bash
pip install \
langgraph \
langchain \
langchain-community \
sentence-transformers \
faiss-cpu \
pypdf \
transformers \
torch
```

---

# Частина 1. State

```python
from typing import TypedDict

class RAGState(TypedDict):
    """
    Стан агента.
    """

    question: str

    context: str

    answer: str
```

---

# Частина 2. Завантаження PDF

```python
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(
    "2408.01747.pdf"
)

documents = loader.load()
```

---

# Частина 3. Розбиття на чанки

Заповніть пропуски.

```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=__________,
    chunk_overlap=__________
)

chunks = splitter.split_documents(
    documents
)
```

---

# Частина 4. Створення ембедінгів

```python
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)
```

---

# Частина 5. Створення векторної бази

Заповніть один пропуск.

```python
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    chunks,
    __________
)
```

---

# Частина 6. Retriever

```python
retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 3
    }
)
```

---

# Частина 7. Ініціалізація Qwen

```python
from transformers import pipeline

llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=300
)
```

---

# Частина 8. Функція виклику LLM

Заповніть один пропуск.

```python
def ask_llm(prompt: str):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    response = llm(
        __________,
        return_full_text=False
    )

    return response[0]["generated_text"]
```

---

# Частина 9. Node 1 — Retrieval

Заповніть пропуски.

```python
def retrieve_documents(state: RAGState):
    """
    Пошук релевантних фрагментів статті.
    """

    docs = retriever.invoke(
        __________
    )

    context = "\n\n".join(
        [
            d.page_content
            for d in docs
        ]
    )

    return {
        "context": context
    }
```

---

# Частина 10. Node 2 — Answer Generation

Заповніть один пропуск.

```python
def generate_answer(state: RAGState):
    """
    Генерує відповідь на основі
    знайдених фрагментів статті.
    """

    prompt = f"""
Answer the question using ONLY the context.

Question:
{state['question']}

Context:
{state['context']}
"""

    answer = __________(prompt)

    return {
        "answer": answer
    }
```

---

# Частина 11. Побудова графа

Заповніть пропуски.

```python
from langgraph.graph import (
    StateGraph,
    START,
    END
)

builder = StateGraph(RAGState)

builder.add_node(
    "retrieve",
    retrieve_documents
)

builder.add_node(
    "answer",
    generate_answer
)

builder.add_edge(
    START,
    __________
)

builder.add_edge(
    __________,
    __________
)

builder.add_edge(
    __________,
    END
)

graph = builder.compile()
```

---

# Частина 12. Запуск

```python
result = graph.invoke(
    {
        "question":
        "What are the major milestones in machine learning?"
    }
)

print(result["answer"])
```

### Очікувана архітектура

```text
START
  ↓
Rewrite Question
  ↓
Retrieve Documents
  ↓
Generate Answer
  ↓
END
```


In [1]:
# # Silent installation or update
#
# # Clean cache
# !python3 -m pip cache purge -q
#
# # Force updating
# package_update = [
#     "pip",
#     "scikit-learn",
#     "pandas",
#     "kagglehub",
#     "kagglesdk",
# ]
#
# for package_name in package_update:
#     !bash -c "python3 -m pip install -U '{package_name}' -q"
#
# # Install missing packages
# package_array = [
#     "jinja2",
#     "ipywidgets",
#     "nbformat",
#     "kagglehub[pandas-datasets]",
#     "numpy",
#     "matplotlib",
#     "scipy",
#     "statsmodels",
#     "joblib",
#     "torch",
#     "transformers",
#     "pypdf",
#     "unstructured",
#     "unstructured[pdf]",
#     "langchain-unstructured",
#     "langchain_text_splitters",
#     "langchain_huggingface",
#     "sentence_transformers",
#     "chromadb",
#     "langchain-chroma",
#     "typing",
#     "langchain_unstructured",
# ]
#
# for package_name in package_array:
#     !bash -c "python3 -m pip show '{package_name}' > /dev/null 2>&1 || python3 -m pip install -U '{package_name}' -q"


In [2]:
# # Synchronization with remote source
#
# import shutil
# from pathlib import Path
#
# # Input data
hm_version = 25
#
# # Solution
# git_project_url = f"https://github.com/BogdanPinchuk/DataScience-PBY_HW{hm_version}.git"
# main_file_name = f"Bohdan_Pinchuk_DS_HW{hm_version}.ipynb"
#
# # upload all files
# current_path = !pwd
# current_path = current_path[0]
# parent_path = !dirname "$current_path"
# parent_path = parent_path[0]
# temp_path = f"{parent_path}/temp"
#
# # Clone data
# !rm -rf "$temp_path"
# !git clone "$git_project_url" "$temp_path"
#
# source = Path(temp_path)
# destination = Path(current_path)
# exclude = {main_file_name, ".git", ".idea"}
#
# for item in source.iterdir():
#     if item.name in exclude:
#         continue
#
#     target = destination / item.name
#     if item.is_dir():
#         shutil.copytree(item, target, dirs_exist_ok=True)
#     else:
#         shutil.copy2(item, target)
#
# # Clean temp folder
# !rm -rf "$temp_path"

# ✅ Крок: Частина 1. State

In [3]:
# State

from typing import TypedDict


# Input data

# Solution
class RAGState(TypedDict):
    """
    Стан агента
    """
    question: str
    question_ua: str
    context: str
    answer: str
    answer_ua: str

# Print results


# ✅ Крок: Частина 2. Завантаження PDF

In [5]:
# Load document

import requests
import apps.reporter as rpt
from pathlib import Path
from langchain_unstructured.document_loaders import UnstructuredLoader

# Input data
df_file_name = "Файл: Classical Machine Learning"
load_file = True
file_name = "resources/сlassical_machine_learning.pdf"
file_url = "https://arxiv.org/pdf/2408.01747.pdf"

# Solution
file_path = Path(file_name)

# try to load file
if load_file and file_path.exists():
    loader = UnstructuredLoader(file_path, strategy="fast", languages=["ukr"])
else:
    response = requests.get(file_url, stream=True)
    # save file
    if response.status_code == 200:
        with open(file_path, "wb") as current_file:
            current_file.write(response.content)
    loader = UnstructuredLoader(file_path, strategy="fast", languages=["ukr"])

docs = loader.load()

rp = rpt.Reporter()
rp.add_item("Розмір документа", str(len(docs)))

# Print results
rp.print_pd_report(df_file_name)

Attribute,Result
Розмір документа,927
